In [1]:
from model import get_model

model = get_model(num_classes=10)

In [2]:
from dataset import get_dataloaders

train_loader, test_loader, class_names = get_dataloaders(
    "C:/Users/Tekin/Downloads/archive/dataset",
    batch_size=32,
    img_size=224
)

print(type(train_loader))
print(type(test_loader))
print(class_names)

Classes: ['Anti-aircraft', 'Armored combat support vehicles', 'Armored personnel carriers', 'Infantry fighting vehicles', 'Light armored vehicles', 'Mine-protected vehicles', 'Prime movers and trucks', 'Self-propelled artillery', 'light utility vehicles', 'tanks']
Train samples: 10414, Test samples: 3719
<class 'torch.utils.data.dataloader.DataLoader'>
<class 'torch.utils.data.dataloader.DataLoader'>
['Anti-aircraft', 'Armored combat support vehicles', 'Armored personnel carriers', 'Infantry fighting vehicles', 'Light armored vehicles', 'Mine-protected vehicles', 'Prime movers and trucks', 'Self-propelled artillery', 'light utility vehicles', 'tanks']


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)

cuda


In [4]:
print(train_loader)
print(type(train_loader))

<class 'torch.utils.data.dataloader.DataLoader'>


In [5]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.1


In [ ]:
# Unfreeze last 3 blocks
for param in model.features[-3:].parameters():
    param.requires_grad = True

# Recreate optimizer so it includes the newly unfrozen parameters
optimizer = torch.optim.Adam([
    {'params': model.features[-3:].parameters(), 'lr': 1e-5},  # unfrozen layers — small lr
    {'params': model.classifier.parameters(), 'lr': 1e-4}       # classifier — normal lr
])

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

num_epochs = 20
best_acc = 0.0

In [7]:

for param in model.features[-3:].parameters():
    param.requires_grad = True

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)


    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total

    scheduler.step()

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}%")



NameError: name 'criterion' is not defined